# Bài thực hành 3 — Mini Data Warehouse với Olist
**Học phần:** Khám phá tri thức và khai thác dữ liệu (GCOS108)  
**Chủ đề:** Data Warehouse, ETL, OLAP và Mining-Ready Dataset

## Dataset
Brazilian E-Commerce Public Dataset by Olist (Kaggle):  
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data

### File bắt buộc trong thư mục `data/`
- `olist_customers_dataset.csv`
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_products_dataset.csv`
- `olist_order_payments_dataset.csv`
- `olist_order_reviews_dataset.csv`
- `olist_sellers_dataset.csv`

> Mục tiêu cuối buổi: **Raw CSV → ETL → Star Schema → OLAP → Customer Mining-Ready Dataset**

## Thông tin nhóm
- Nhóm: **TODO**
- Thành viên: **TODO**
- Phân công: **TODO**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

REQUIRED_FILES = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_sellers_dataset.csv",
]

missing = [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "Thiếu file trong data/:\n- " + "\n- ".join(missing) +
        "\nHãy tải dataset Olist từ Kaggle và đặt các CSV vào thư mục data/."
    )

print("Data folder OK")

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

translation_path = DATA_DIR / "product_category_name_translation.csv"
translation = pd.read_csv(translation_path) if translation_path.exists() else None

TABLES = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers,
}

## A. Source Data Inventory
**TODO:** Với mỗi bảng, in `shape`, cột, 3 dòng đầu. Sau đó điền bảng: *một dòng đại diện cho gì? business key? join với bảng nào? cardinality dự kiến?*

In [ ]:
# TODO A1: khảo sát các bảng nguồn
for name, df in TABLES.items():
    # TODO: in name, shape, columns, head(3)
    pass

### A2. Câu trả lời
**TODO:** Viết nhận xét về khóa và grain của từng bảng. Đặc biệt: `orders` và `order_items` có cùng grain không?

In [ ]:
# TODO A3: kiểm tra uniqueness / duplicate của các khóa quan trọng
# Gợi ý: nunique(), duplicated(), duplicated(subset=[...])

## B. Xác định Grain và thiết kế Star Schema
**Yêu cầu:** Chọn grain cho `FactSales`: **1 row = 1 order item**.

**TODO:** Vẽ/ghi schema gồm FactSales + DimCustomer + DimProduct + DimSeller + DimDate. Nêu measure và foreign keys.

### B1. Trả lời
- Grain FactSales: **TODO**
- Business key của FactSales: **TODO**
- Measures: **TODO**
- Vì sao không chọn 1 row = 1 order? **TODO**

## C. Row Explosion Lab
Thực hiện join `orders → items → payments` theo `order_id`. So sánh số dòng và tổng `price` trước/sau join.

In [ ]:
# TODO C1
# tmp = ...
# print số dòng items, tmp
# so sánh items["price"].sum() và tmp["price"].sum()

### C2. Giải thích
**TODO:** Vì sao join hợp lệ về SQL nhưng có thể gây double counting? Đề xuất cách tránh.

## D. ETL và Data Quality

In [ ]:
# TODO D1: parse datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
# TODO: pd.to_datetime(..., errors="coerce")

In [ ]:
# TODO D2: tạo delivery_days, delivery_delay_days, late_delivery

In [ ]:
# TODO D3: lập missing report cho orders/items/products/customers

### D4. Business rules
**TODO:** Kiểm tra và nhận xét ít nhất 3 rule, ví dụ `price >= 0`, `freight_value >= 0`, delivered date không trước purchase date.

## E. Tạo Dimension Tables + Surrogate Keys

In [ ]:
# TODO E1: DimCustomer
# Cột tối thiểu: customer_key, customer_id, customer_unique_id, customer_city, customer_state

In [ ]:
# TODO E2: DimProduct
# Nếu có translation, có thể thêm product_category_name_english

In [ ]:
# TODO E3: DimSeller

In [ ]:
# TODO E4: DimDate từ min/max order_purchase_timestamp

### E5. Trả lời
Phân biệt `customer_id`, `customer_unique_id` và `customer_key`. Vì sao cần surrogate key? **TODO**

## F. Xây dựng FactSales và Reconciliation

In [ ]:
# TODO F1: join items + orders + mapping các surrogate keys
# Giữ grain = (order_id, order_item_id)

In [ ]:
# TODO F2: quality checks
# - duplicate (order_id, order_item_id)
# - null foreign keys
# - describe price/freight

In [ ]:
# TODO F3: reconciliation
# So sánh source items tổng price/freight với FactSales

## G. Load DuckDB và OLAP

In [ ]:
# Nếu duckdb chưa có, bỏ comment dòng dưới:
# %pip install duckdb -q
import duckdb
con = duckdb.connect(str(OUTPUT_DIR / "olist_dw.duckdb"))
# TODO: register/create FactSales và Dimensions

In [ ]:
# TODO G2: OLAP 1 - Revenue theo year/month
query_monthly = """
-- TODO SQL
"""
# monthly = con.execute(query_monthly).df()
# monthly.head()

In [ ]:
# TODO G3: OLAP 2 - Revenue + distinct orders theo customer_state

In [ ]:
# TODO G4: OLAP 3 - Top product category theo revenue và items_sold

In [ ]:
# TODO G5: Slice/Dice - ví dụ year=2017, state IN ('SP','RJ')

## H. FactDelivery và Fact Constellation

In [ ]:
# TODO H1: tạo FactDelivery với grain = 1 order
# columns gợi ý: order_id, customer_key, purchase_date_key, delivery_days, delivery_delay_days, late_delivery

### H2. Trả lời
FactSales và FactDelivery có thể chia sẻ dimension nào? **TODO**

## I. Customer Mining-Ready Dataset

In [ ]:
# TODO I1-I3:
# Grain = 1 row / customer_unique_id
# Features tối thiểu: total_orders, total_items, total_spending, avg_item_price, avg_freight, days_since_last_purchase
# Mở rộng: avg_review_score, late_delivery_rate

In [ ]:
# TODO I4: export
# customer_features.to_csv(OUTPUT_DIR / "customer_mining_ready.csv", index=False)

## J. Phản tư cuối buổi
Viết 200-300 từ trả lời:
1. Grain, Fact, Dimension của thiết kế.
2. Một lỗi thiết kế/ETL đã phát hiện.
3. Vì sao Data Warehouse khác Mining-Ready Dataset?
4. Nếu dùng dữ liệu này để dự đoán khách hàng quay lại, prediction time nên xác định thế nào?

## Checklist
- [ ] Restart & Run All không lỗi
- [ ] Có reconciliation
- [ ] Có 4 OLAP queries
- [ ] Có Star Schema
- [ ] Có `olist_dw.duckdb`
- [ ] Có `customer_mining_ready.csv`
- [ ] Có phần phản tư